In [ ]:
# ==============================================================================
# Cell 1: Environment Setup, Data Loading & SMOTE Balancing (Seed 1234)
# ==============================================================================
import os
import glob
import gc
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter, MaxNLocator

from sklearn.utils import resample
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score, roc_curve, average_precision_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE

import rasterio
from rasterio.warp import reproject, Resampling
import pyproj

from pycaret.classification import setup, create_model, save_model, load_model, predict_model

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Calibri', 'DejaVu Sans', 'Liberation Sans', 'Arial', 'Helvetica'],
})

print("⏳ Step 1: Loading Data and Balancing Classes...")

train_data_df = pd.read_csv("train (1).csv")
test_data_df = pd.read_csv("test.csv")

scenarios = {
    "ninefeatures": [
        'Lithology', 'DEM', 'Slope', 'Aspect', 'Plancurv', 'Procurv',
        'SPI', 'NDVI', 'Rainfall', 'Scar'
    ],
    "elevenfeatures": [
        'Lithology', 'DEM', 'Slope', 'Aspect', 'Plancurv', 'Procurv',
        'SPI', 'NDVI', 'Rainfall', 'Angle of friction', 'Cohesion (T/m2)', 'Scar'
    ]
}

data_dict = {}

for s_name, features in scenarios.items():
    print(f"🔄 Processing {s_name}...")

    train_df = train_data_df[features].dropna()
    test_df = test_data_df[features].dropna()

    minority = train_df[train_df['Scar'] == 1]
    majority = train_df[train_df['Scar'] == 0]

    # 🎯 ใช้ Seed 1234 เพื่อให้ข้อมูลเหมือนเดิมเป๊ะ
    majority_downsampled = resample(majority, replace=False, n_samples=2 * len(minority), random_state=1234)
    balanced_partial = pd.concat([minority, majority_downsampled])

    X_train, y_train = balanced_partial.drop('Scar', axis=1), balanced_partial['Scar']

    X_res, y_res = SMOTE(random_state=1234).fit_resample(X_train, y_train)

    balanced_train = pd.concat([pd.DataFrame(X_res, columns=X_train.columns), pd.Series(y_res, name='Scar')], axis=1)
    data_dict[s_name] = {"train": balanced_train, "test": test_df}

print("✅ Data Preparation Completed!")


In [ ]:
# ==============================================================================
# Cell 2: Machine Learning Model Training (PyCaret)
# * Note: This cell demonstrates the training process.
# It saves to 'results_reproduced/' to avoid overwriting original pre-trained models.
# ==============================================================================
print("⏳ Step 2: Training ML Models (PyCaret)...")
target = 'Scar'
MODELS_TO_TRAIN = ['et', 'catboost', 'rf', 'xgboost', 'lightgbm', 'knn', 'gbc', 'ada', 'dt']

for s_name, data in data_dict.items():
    # 🎯 เซฟลงโฟลเดอร์แยกต่างหากเพื่อไม่ให้ทับไฟล์ .pkl ของแท้
    MODELS_DIR = Path(f"results_reproduced/{s_name}/models")
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    print(f"\n🚀 Training Models for {s_name} -> Saving to {MODELS_DIR}...")

    clf_setup = setup(
        data=data["train"], target=target, session_id=1234,
        normalize=True, train_size=0.8, fold=10, verbose=False
    )

    for m_id in MODELS_TO_TRAIN:
        print(f"   -> Training {m_id}...")
        model = create_model(m_id, verbose=False)
        model_class_name = type(model.steps[-1][1]).__name__ if hasattr(model, 'steps') else type(model).__name__
        save_model(model, str(MODELS_DIR / f"top_{model_class_name}"))

print("\n✅ All Models Trained and Saved in results_reproduced/!")


In [ ]:
# ==============================================================================
# Cell 3: Machine Learning Model Evaluation (Optimal Threshold)
# ==============================================================================
print("⏳ Step 3: Evaluating Pre-trained ML Models and Calculating Metrics...")
ml_results = []

for s_name, data in data_dict.items():
    # 🎯 ดึงไฟล์โมเดลของแท้จากโฟลเดอร์ pretrained_models
    model_paths = sorted(glob.glob(f"pretrained_models/{s_name}/models/*.pkl"))
    test_df = data["test"]
    scenario_label = "S1 (9 Factors)" if "nine" in s_name else "S2 (11 Factors)"

    if not model_paths:
        print(f"⚠️ Warning: No pre-trained models found in pretrained_models/{s_name}/models/")
        continue

    for mp in model_paths:
        model = load_model(mp.replace(".pkl", ""), verbose=False)
        model_name = type(model.steps[-1][1]).__name__ if hasattr(model, 'steps') else type(model).__name__

        preds = predict_model(model, data=test_df, raw_score=True)
        y_true = preds['Scar']
        prob_col = [c for c in preds.columns if '1' in c and 'score' in c.lower()][0]
        y_prob = preds[prob_col]

        roc_auc = roc_auc_score(y_true, y_prob)
        pr_auc = average_precision_score(y_true, y_prob)

        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        optimal_threshold = thresholds[np.argmax(tpr - fpr)]

        y_pred = (y_prob >= optimal_threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        specificity = tn / (tn + fp)

        ml_results.append({
            "Scenario": scenario_label,
            "Model": model_name.replace("Classifier", "").strip(),
            "Threshold (Copt)": round(optimal_threshold, 4),
            "Accuracy": round(accuracy_score(y_true, y_pred), 4),
            "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
            "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
            "Specificity": round(specificity, 4),
            "F1-Score": round(f1_score(y_true, y_pred, zero_division=0), 4),
            "MCC": round(matthews_corrcoef(y_true, y_pred), 4),
            "ROC-AUC": round(roc_auc, 4),
            "PR-AUC": round(pr_auc, 4)
        })

ml_results_df = pd.DataFrame(ml_results)
ml_results_df.to_csv("ML_Metrics_Summary_github.csv", index=False)
print("✅ ML Evaluation Completed!")


In [ ]:
# ==============================================================================
# Cell 4: FR Validation & Generating Table 3 and Table 4 (Lat/Long Input & Auto Threshold)
# ==============================================================================
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    matthews_corrcoef, roc_auc_score, roc_curve, average_precision_score, confusion_matrix
)
from pathlib import Path

# 🎯 บังคับให้ Pandas แสดงผลตารางเต็ม ไม่มีการตัดทอน
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

print("⏳ Step 4: Evaluating FR Models (Lat/Long Input) and Generating Final Tables...")

csv_path = "test.csv"
df_test = pd.read_csv(csv_path)

# ดึงชื่อคอลัมน์พิกัด
lat_candidates = ['Lat', 'latitude', 'LAT', 'Y', 'utm_y']
lon_candidates = ['Long', 'Longitude', 'LONG', 'longitude', 'X', 'utm_x']
lat_col = next((c for c in lat_candidates if c in df_test.columns), None)
lon_col = next((c for c in lon_candidates if c in df_test.columns), None)

lsi_paths = {"S1 (9 Factors)": "exclude1.tif", "S2 (11 Factors)": "include1.tif"}
fr_results = []

def extract_lsi_from_latlon(raster_path, df, lat_col, long_col):
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

        lons = df[long_col].values
        lats = df[lat_col].values

        if raster_crs and raster_crs != "EPSG:4326":
            xs, ys = transform('EPSG:4326', raster_crs, lons, lats)
        else:
            xs, ys = lons, lats

        coords = list(zip(xs, ys))
        values = []
        for val in src.sample(coords):
            v = val[0]
            if nodata is not None and np.isclose(v, nodata):
                values.append(np.nan)
            elif np.isclose(v, -3.4028234663852886e+38) or v > 1e15 or v < -1e15:
                values.append(np.nan)
            else:
                values.append(v)

    return np.array(values)

for s_name, lsi_path in lsi_paths.items():
    if not Path(lsi_path).exists():
        print(f"⚠️ ไม่พบไฟล์ {lsi_path}")
        continue

    # ดึง LSI จากพิกัด Lat/Long แทนการคำนวณ Raster ทั้งแผ่น
    lsi_scores = extract_lsi_from_latlon(lsi_path, df_test, lat_col, lon_col)
    y_true = df_test['Scar'].values

    valid_mask = ~np.isnan(lsi_scores) & ~np.isnan(y_true)
    y_true_clean = y_true[valid_mask]
    lsi_scores_clean = lsi_scores[valid_mask]

    roc_auc = roc_auc_score(y_true_clean, lsi_scores_clean)
    pr_auc = average_precision_score(y_true_clean, lsi_scores_clean)

    # หา Optimal Threshold (Youden's J)
    fpr, tpr, thresholds = roc_curve(y_true_clean, lsi_scores_clean)
    optimal_threshold = thresholds[np.argmax(tpr - fpr)]

    y_pred = (lsi_scores_clean >= optimal_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_clean, y_pred).ravel()
    specificity = tn / (tn + fp)

    fr_results.append({
        "Scenario": "Scenario 1: 9 causative factors" if "9" in s_name else "Scenario 2: 11 causative factors",
        "Model": "Frequency Ratio (FR)",
        "Threshold (Copt)": round(optimal_threshold, 4),
        "Accuracy": round(accuracy_score(y_true_clean, y_pred), 4),
        "Precision": round(precision_score(y_true_clean, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true_clean, y_pred, zero_division=0), 4),
        "Specificity": round(specificity, 4),
        "F1-Score": round(f1_score(y_true_clean, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_true_clean, y_pred), 4),
        "ROC-AUC": round(roc_auc, 4),
        "PR-AUC": round(pr_auc, 4)
    })

fr_results_df = pd.DataFrame(fr_results)

try:
    ml_results_df = pd.read_csv("ML_Metrics_Summary_github.csv")
except:
    try:
        ml_results_df = pd.DataFrame(ml_results)
    except NameError:
        print("⚠️ ไม่พบข้อมูล ML โปรดรัน Cell 3 ก่อน!")
        ml_results_df = pd.DataFrame()

if not ml_results_df.empty:
    ml_results_df['Scenario'] = ml_results_df['Scenario'].replace({
        "S1 (9 Factors)": "Scenario 1: 9 causative factors",
        "S2 (11 Factors)": "Scenario 2: 11 causative factors"
    })

all_results = pd.concat([fr_results_df, ml_results_df]).reset_index(drop=True)

if not all_results.empty:
    table_3_s1 = all_results[all_results['Scenario'] == "Scenario 1: 9 causative factors"].copy()
    table_3_s1['sort_key'] = np.where(table_3_s1['Model'] == 'Frequency Ratio (FR)', 0, 1)
    table_3_s1 = table_3_s1.sort_values(by=['sort_key', 'Model']).drop(columns=['sort_key', 'ROC-AUC', 'PR-AUC'])

    table_3_s2 = all_results[all_results['Scenario'] == "Scenario 2: 11 causative factors"].copy()
    table_3_s2['sort_key'] = np.where(table_3_s2['Model'] == 'Frequency Ratio (FR)', 0, 1)
    table_3_s2 = table_3_s2.sort_values(by=['sort_key', 'Model']).drop(columns=['sort_key', 'ROC-AUC', 'PR-AUC'])

    table_3 = pd.concat([table_3_s1, table_3_s2]).reset_index(drop=True)

    try:
        from IPython.display import display
        print("\n" + "="*90)
        print("TABLE 3: Threshold-dependent classification performance")
        print("="*90)
        display(table_3)
    except:
        print(table_3)

    all_results_clean = all_results.drop_duplicates(subset=['Scenario', 'Model'], keep='last')
    table_4_pivot = all_results_clean.pivot_table(index='Model', columns='Scenario', values=['ROC-AUC', 'PR-AUC'])

    table_4 = pd.DataFrame({
        'Model': table_4_pivot.index,
        'ROC-AUC (S1)': table_4_pivot[('ROC-AUC', 'Scenario 1: 9 causative factors')],
        'ROC-AUC (S2)': table_4_pivot[('ROC-AUC', 'Scenario 2: 11 causative factors')],
        'PR-AUC (S1)': table_4_pivot[('PR-AUC', 'Scenario 1: 9 causative factors')],
        'PR-AUC (S2)': table_4_pivot[('PR-AUC', 'Scenario 2: 11 causative factors')]
    }).reset_index(drop=True)

    table_4['ΔROC-AUC (S2-S1)'] = table_4['ROC-AUC (S2)'] - table_4['ROC-AUC (S1)']
    table_4['ΔPR-AUC (S2-S1)'] = table_4['PR-AUC (S2)'] - table_4['PR-AUC (S1)']

    table_4['sort_key'] = np.where(table_4['Model'] == 'Frequency Ratio (FR)', 0, 1)
    table_4 = table_4.sort_values(by=['sort_key', 'Model']).drop(columns=['sort_key']).reset_index(drop=True)

    try:
        print("\n" + "="*90)
        print("TABLE 4: Threshold-independent discrimination performance")
        print("="*90)
        display(table_4)
    except:
        print(table_4)

    table_3.to_csv("Table3_Metrics_github.csv", index=False)
    table_4.to_csv("Table4_AUCs_github.csv", index=False)


In [ ]:
# ==============================================================================
# Cell 5: Feature Importance Analysis (Layer Cake Plot) - github version
# ==============================================================================
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pycaret.classification import load_model

# ============================================================
# 1. ระบบจัดการฟอนต์มาตรฐานเดียวกันทั้งโปรเจกต์ (Typography System)
# =====================================================================
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Calibri', 'DejaVu Sans', 'Liberation Sans', 'Arial', 'Helvetica'],
})

FONT_TITLE = 16
FONT_AXIS_TITLE = 14
FONT_TICK = 12
FONT_LEGEND = 12
FONT_BAR_LABEL = 11

# ============================================================
# 2. ตั้งค่าชื่อ ฟีเจอร์ และสีของโมเดล (มาตรฐานสากล)
# ============================================================
FEATURE_MAPPING = {
    "Lithology": "Lithology",
    "DEM": "Elevation",
    "Slope": "Slope",
    "Aspect": "Aspect",
    "Plancurv": "Plan Curvature",
    "Procurv": "Profile Curvature",
    "SPI": "SPI",
    "NDVI": "NDVI",
    "Rainfall": "Rainfall",
    "Angle of friction": "Internal Friction Angle",
    "Cohesion (T/m2)": "Cohesion"
}

MODEL_COLORS = {
    "Extra Trees": "#1f77b4",        # Blue
    "CatBoost": "#ff7f0e",           # Orange
    "Random Forest": "#2ca02c",      # Green
    "XGBoost": "#d62728",            # Red
    "LightGBM": "#9467bd",           # Purple
    "Gradient Boosting": "#8c564b",  # Brown
    "AdaBoost": "#e377c2",           # Pink
    "Decision Tree": "#7f7f7f",      # Gray
    "K-Nearest Neighbors": "#bcbd22"
}

TARGET_MODELS = list(MODEL_COLORS.keys())

MODEL_KEYWORDS = {
    "extratrees": "Extra Trees", "et": "Extra Trees",
    "catboost": "CatBoost",
    "randomforest": "Random Forest", "rf": "Random Forest",
    "xgboost": "XGBoost", "xgb": "XGBoost",
    "lightgbm": "LightGBM", "lgbm": "LightGBM", "lgb": "LightGBM",
    "gradientboosting": "Gradient Boosting", "gbc": "Gradient Boosting",
    "adaboost": "AdaBoost", "ada": "AdaBoost",
    "decisiontree": "Decision Tree", "dt": "Decision Tree",
    "kneighbors": "K-Nearest Neighbors", "knn": "K-Nearest Neighbors"
}

# ============================================================
# 3. ฟังก์ชันดึงข้อมูลจากโฟลเดอร์โมเดล (ดึงจาก pretrained_models)
# ============================================================
def identify_model_type(file_path):
    name_clean = file_path.stem.lower().replace("_", "").replace("-", "")
    for kw, std_name in MODEL_KEYWORDS.items():
        if kw in name_clean:
            return std_name
    return None

def get_final_estimator(model):
    if hasattr(model, "steps") and len(model.steps) > 0:
        return model.steps[-1][1]
    return model

def extract_importance_from_dir(dir_name, expected_features_count):
    # 🎯 บังคับให้ดึงจาก pretrained_models ตามโครงสร้าง GitHub ที่คุณต้องการ
    model_dir = Path(f"pretrained_models/{dir_name}/models")

    if not model_dir.exists():
        print(f"❌ ไม่พบโฟลเดอร์โมเดล: {model_dir}")
        return pd.DataFrame()

    all_pkl_files = sorted(model_dir.glob("*.pkl"))
    selected_files = {}

    for f in all_pkl_files:
        m_type = identify_model_type(f)
        if m_type in TARGET_MODELS:
            if m_type not in selected_files or f.stat().st_mtime > selected_files[m_type].stat().st_mtime:
                selected_files[m_type] = f

    importance_dfs = []

    if expected_features_count == 9:
        raw_features = ["Lithology", "DEM", "Slope", "Aspect", "Plancurv", "Procurv", "SPI", "NDVI", "Rainfall"]
    else:
        raw_features = ["Lithology", "DEM", "Slope", "Aspect", "Plancurv", "Procurv", "SPI", "NDVI", "Rainfall", "Angle of friction", "Cohesion (T/m2)"]

    mapped_features = [FEATURE_MAPPING[f] for f in raw_features]

    for m_name, m_file in selected_files.items():
        try:
            model = load_model(str(m_file.with_suffix("")), verbose=False)
            estimator = get_final_estimator(model)

            if hasattr(estimator, "feature_importances_"):
                importance = np.asarray(estimator.feature_importances_, dtype=float).ravel()
            elif hasattr(estimator, "coef_"):
                coef = np.asarray(estimator.coef_, dtype=float)
                importance = np.abs(coef[0]) if coef.ndim == 2 else np.abs(coef)
                importance = np.asarray(importance, dtype=float).ravel()
            else:
                continue

            if len(importance) != expected_features_count:
                continue

            importance = np.abs(importance)
            if importance.sum() == 0: continue

            # แปลงให้เป็นสัดส่วนเปอร์เซ็นต์ (0-100)
            percent_importance = (importance / importance.sum()) * 100

            df = pd.DataFrame({
                "Feature": mapped_features,
                "Importance_Pct": percent_importance,
                "Model": m_name
            })
            importance_dfs.append(df)

        except Exception:
            pass

    if not importance_dfs:
        return pd.DataFrame()

    return pd.concat(importance_dfs, ignore_index=True)


# ============================================================
# 4. โหลดและจัดการ Pivot Data
# ============================================================
print("⏳ กำลังประมวลผลข้อมูล 9 Features...")
df_9 = extract_importance_from_dir("ninefeatures", 9)
print("⏳ กำลังประมวลผลข้อมูล 11 Features...")
df_11 = extract_importance_from_dir("elevenfeatures", 11)

if df_9.empty or df_11.empty:
    raise RuntimeError("❌ ไม่สามารถดึงข้อมูลได้ครบถ้วน โปรดตรวจสอบเส้นทางโฟลเดอร์โมเดล (pretrained_models/)")

# 🎯 ใช้การคำนวณแบบต้นฉบับของคุณ
num_models_9 = df_9["Model"].nunique()
pivot_9 = df_9.pivot_table(index="Feature", columns="Model", values="Importance_Pct", fill_value=0) / num_models_9

num_models_11 = df_11["Model"].nunique()
pivot_11 = df_11.pivot_table(index="Feature", columns="Model", values="Importance_Pct", fill_value=0) / num_models_11

# เรียงลำดับ Feature จากผลรวมเปอร์เซ็นต์มากไปน้อย
pivot_9 = pivot_9.loc[pivot_9.sum(axis=1).sort_values(ascending=False).index]
pivot_11 = pivot_11.loc[pivot_11.sum(axis=1).sort_values(ascending=False).index]


# ============================================================
# 5. สร้างกราฟ 2 กรอบ (Layer Cake) พร้อมตัวเลขด้านบนสุดของบาร์
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 15), dpi=300)

def plot_stacked_bar_with_top_label(pivot_df, ax, title_text):
    # ป้องกัน error กรณีบางโมเดลหายไป
    colors = [MODEL_COLORS[model] for model in pivot_df.columns if model in MODEL_COLORS]
    pivot_df.plot(kind="bar", stacked=True, ax=ax, color=colors, width=0.75, edgecolor='black', linewidth=0.5)

    ax.set_title(title_text, loc="left", fontweight="bold", fontsize=FONT_TITLE, pad=15)
    ax.set_ylabel("Combined Importance (%)", fontweight="bold", fontsize=FONT_AXIS_TITLE)
    ax.tick_params(axis='x', rotation=25, labelsize=FONT_TICK)
    ax.tick_params(axis='y', labelsize=FONT_TICK)
    ax.get_legend().remove()
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    max_height = pivot_df.sum(axis=1).max()
    ax.set_ylim(0, max_height * 1.15)

    totals = pivot_df.sum(axis=1)
    for i, (feature_name, total) in enumerate(totals.items()):
        ax.text(i, total + (max_height * 0.02), f"{total:05.2f}%", ha='center', va='bottom',
                fontsize=FONT_BAR_LABEL, fontweight='bold', color='black')

plot_stacked_bar_with_top_label(pivot_9, axes[0], "(a)")
axes[0].set_xlabel("")

plot_stacked_bar_with_top_label(pivot_11, axes[1], "(b)")
axes[1].set_xlabel("Causative Factors", fontweight="bold", fontsize=FONT_AXIS_TITLE, labelpad=10)


# ============================================================
# 6. สร้าง Global Legend และ Save ภาพ (ใช้ชื่อ _github ป้องกันการทับ)
# ============================================================
all_used_models = list(set(pivot_9.columns.tolist() + pivot_11.columns.tolist()))
all_used_models.sort()

legend_patches = [mpatches.Patch(color=MODEL_COLORS[m], label=m) for m in all_used_models]

plt.tight_layout()
fig.subplots_adjust(bottom=0.12, hspace=0.35)

fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=FONT_LEGEND,
           title="Machine Learning Classifiers", title_fontproperties={'weight':'bold', 'size': FONT_LEGEND},
           bbox_to_anchor=(0.5, -0.04), frameon=True)

output_path = "FINAL_COMBINED_FEATURE_IMPORTANCE_CAKE_github.png"
# DPI 300 เพียงพอแล้วสำหรับสิ่งพิมพ์ (DPI 1000 อาจจะทำให้ไฟล์ใหญ่เกินไปและค้างได้)
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"\n✅ บันทึกรูปกราฟเรียบร้อยแล้วที่: {output_path}")

try:
    from IPython.display import display
    display(fig)
except:
    plt.show()


In [ ]:
# ==============================================================================
# Cell 6: Machine Learning Spatial Susceptibility Mapping
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path
import glob
import gc
from sklearn.cluster import KMeans
from pycaret.classification import load_model, predict_model
from matplotlib.ticker import FuncFormatter, MaxNLocator

try:
    from IPython import get_ipython
    from IPython.display import display
    if get_ipython() is not None:
        get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Calibri', 'DejaVu Sans', 'Liberation Sans', 'Arial', 'Helvetica'],
})

TEST_PATH = "test.csv"
test_data_df = pd.read_csv(TEST_PATH)

# 🎯 ดึงจากโฟลเดอร์ pretrained_models
DIR_S1 = Path("pretrained_models/ninefeatures/models")
DIR_S2 = Path("pretrained_models/elevenfeatures/models")

def get_valid_models(models_dir):
    if not models_dir.exists():
        print(f"Directory not found: {models_dir}")
        return []
    all_pkls = sorted(glob.glob(str(models_dir / "*.pkl")))
    model_files = [f for f in all_pkls if "latest" not in f and "pipeline" not in f and "transformer" not in f]
    top_files = sorted([f for f in model_files if "top" in Path(f).name])
    return top_files if top_files else model_files

models_s1_raw = get_valid_models(DIR_S1)
models_s2_raw = get_valid_models(DIR_S2)

if not models_s1_raw or not models_s2_raw:
    raise RuntimeError("❌ ไม่พบไฟล์โมเดลครบทั้ง 2 Scenarios กรุณาตรวจสอบโฟลเดอร์ pretrained_models/")

MODEL_KEYWORDS = {
    "extratrees": "Extra Trees", "et": "Extra Trees",
    "catboost": "CatBoost",
    "randomforest": "Random Forest", "rf": "Random Forest",
    "xgboost": "XGBoost", "xgb": "XGBoost",
    "lightgbm": "LightGBM", "lgbm": "LightGBM", "lgb": "LightGBM",
    "kneighbors": "K-Nearest Neighbors", "knn": "K-Nearest Neighbors",
    "gradientboosting": "Gradient Boosting", "gbc": "Gradient Boosting",
    "adaboost": "AdaBoost", "ada": "AdaBoost",
    "decisiontree": "Decision Tree", "dt": "Decision Tree"
}

def get_clean_model_name(file_name):
    name_clean = file_name.lower().replace("_", "").replace("-", "")
    for kw, std_name in MODEL_KEYWORDS.items():
        if kw in name_clean: return std_name
    return file_name

TARGET_ORDER = [
    "Extra Trees", "CatBoost", "Random Forest", "LightGBM", "XGBoost",
    "K-Nearest Neighbors", "Gradient Boosting", "AdaBoost", "Decision Tree"
]

models_s1, models_s2 = [], []
for target in TARGET_ORDER:
    match_s1 = next((s for s in models_s1_raw if get_clean_model_name(Path(s).name) == target), None)
    match_s2 = next((s for s in models_s2_raw if get_clean_model_name(Path(s).name) == target), None)
    if match_s1 and match_s2:
        models_s1.append(match_s1)
        models_s2.append(match_s2)

n_models = len(models_s1)
print(f"✅ Found {n_models} matched model pairs for plotting.")

lat_col, lon_col = 'Lat', 'Long'
class_labels = ["Very Low", "Low", "Moderate", "High", "Very High"]
hex_colors = ["#2b83ba", "#abdda4", "#ffffbf", "#fdae61", "#d7191c"]
cmap_susceptibility = ListedColormap(hex_colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], 5)

def format_lon(x, pos): return f"{x:.2f}°E"
def format_lat(y, pos): return f"{y:.2f}°N"

def plot_single_map(ax, model_path, test_df, letter_char, scenario_label, model_name):
    model_stem = str(model_path).replace(".pkl", "")
    model = load_model(model_stem, verbose=False)
    predictions = predict_model(model, data=test_df, raw_score=True)

    lat_data, lon_data = predictions[lat_col], predictions[lon_col]
    prob_cols = [c for c in predictions.columns if '1' in c and 'score' in c.lower()]
    if not prob_cols: prob_cols = [c for c in predictions.columns if '1' in c]
    scar_probability = predictions[prob_cols[0]].astype(float)

    X_val = scar_probability.values.reshape(-1, 1)
    try:
        kmeans = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_val)
        centers = np.sort(kmeans.cluster_centers_.flatten())
        jenks_breaks = [0.0] + [(centers[j] + centers[j+1])/2 for j in range(len(centers)-1)] + [1.0]
        jenks_breaks = np.unique(jenks_breaks)
        if len(jenks_breaks) < 6: jenks_breaks = np.linspace(0.0, 1.0, 6)
    except:
        jenks_breaks = np.linspace(0.0, 1.0, 6)

    numeric_natural = pd.cut(scar_probability, bins=jenks_breaks, labels=[0, 1, 2, 3, 4], include_lowest=True).astype(int)
    class_natural = pd.cut(scar_probability, bins=jenks_breaks, labels=class_labels, include_lowest=True)
    class_percentages = (class_natural.value_counts(normalize=True) * 100).reindex(class_labels, fill_value=0.0)

    sc = ax.scatter(
        lon_data, lat_data, c=numeric_natural, cmap=cmap_susceptibility, norm=norm,
        s=10, alpha=0.85, edgecolors="none", rasterized=True
    )

    ax.set_title(f"({letter_char}) {model_name}; {scenario_label}", fontsize=14, fontweight="bold", loc="left", pad=8)
    ax.set_xlabel("Longitude", fontsize=11, fontweight="bold", labelpad=2)
    ax.set_ylabel("Latitude", fontsize=11, fontweight="bold", labelpad=2)

    ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_lat))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.3)
    ax.tick_params(axis="both", labelsize=9)
    ax.set_aspect("equal", adjustable="box")

    y_pos = -0.22
    for idx, (lbl, val) in enumerate(class_percentages.items()):
        x_group = 0.01 + (idx * 0.20)
        ax.plot(x_group, y_pos, marker='o', markersize=9, color=hex_colors[idx],
                markeredgecolor='black', markeredgewidth=0.5, transform=ax.transAxes, clip_on=False)
        ax.text(x_group + 0.035, y_pos, f"{val:.1f}%", transform=ax.transAxes,
                ha='left', va='center', fontsize=10, fontweight='bold', clip_on=False)

    del model, predictions
    gc.collect()
    return sc, class_percentages

chunks = [(0, 4), (4, 8), (8, 9)]
letter_ascii = 97

for chunk_idx, (start_idx, end_idx) in enumerate(chunks):
    if start_idx >= n_models: break

    actual_end_idx = min(end_idx, n_models)
    n_rows = actual_end_idx - start_idx

    if n_rows <= 0: continue

    fig_height = 7.0 if n_rows == 1 else 5.5 * n_rows
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, fig_height), dpi=100)

    if n_rows == 1:
        axes = np.array([axes])

    fig.subplots_adjust(hspace=0.45, wspace=0.30)

    print(f"\n🎨 Drawing ML Map Chunk {chunk_idx + 1} (Models {start_idx+1} to {actual_end_idx})...")
    last_sc = None

    for row, model_idx in enumerate(range(start_idx, actual_end_idx)):
        model_name = get_clean_model_name(Path(models_s1[model_idx]).name)

        ax_left = axes[row, 0]
        letter_left = chr(letter_ascii)
        letter_ascii += 1
        print(f"  -> Plotting [{letter_left}] S1: {model_name}")
        sc_s1, _ = plot_single_map(ax_left, models_s1[model_idx], test_data_df, letter_left, "S1", model_name)

        ax_right = axes[row, 1]
        letter_right = chr(letter_ascii)
        letter_ascii += 1
        print(f"  -> Plotting [{letter_right}] S2: {model_name}")
        sc_s2, _ = plot_single_map(ax_right, models_s2[model_idx], test_data_df, letter_right, "S2", model_name)

        last_sc = sc_s2

    fig.subplots_adjust(bottom=0.10 if n_rows > 1 else 0.20)
    cbar_ax = fig.add_axes([0.35, 0.02, 0.3, 0.015 if n_rows > 1 else 0.03])
    cbar = fig.colorbar(last_sc, cax=cbar_ax, orientation="horizontal", ticks=[0, 1, 2, 3, 4])
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(size=0)
    cbar.set_ticklabels(class_labels)
    cbar.set_label("Landslide Susceptibility Classes", fontsize=14, fontweight="bold", labelpad=10)

    # 🎯 บันทึกไฟล์ด้วยชื่อ _github
    output_filename = f"comparison_map_ML_github_part{chunk_idx + 1}.png"
    plt.savefig(output_filename, dpi=300, bbox_inches="tight")
    print(f"✅ Saved Map: {output_filename}")

    try:
        fig.canvas.draw()
        display(fig)
    except Exception as e:
        raise RuntimeError(f"❌ ไม่สามารถแสดงผลกราฟใน Kernel ได้ เกิดข้อผิดพลาด: {str(e)}")

    plt.close(fig)
    gc.collect()

print("\n🎉 พล็อตแผนที่เปรียบเทียบเสร็จสมบูรณ์ทั้ง 3 ท่อนแล้ว!")


In [ ]:
# ==============================================================================
# Cell 7: Frequency Ratio (FR) Spatial Mapping (Fixed Colors)
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter, MaxNLocator
import rasterio
import pyproj
from pathlib import Path

try:
    from IPython import get_ipython
    from IPython.display import display
    if get_ipython() is not None:
        get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

print("🎨 Drawing Frequency Ratio (FR) Maps...")

file_sc1 = Path("exclude1.tif")
file_sc2 = Path("include1.tif")

def load_geotiff(path):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)
        nodata = src.nodata
        if nodata is not None: data[data == nodata] = np.nan
        data[data < -9000] = np.nan

        bounds = src.bounds
        extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
        crs = src.crs
        return data, extent, crs

if file_sc1.exists() and file_sc2.exists():
    data_sc1, ext_sc1, crs_sc1 = load_geotiff(file_sc1)
    data_sc2, ext_sc2, crs_sc2 = load_geotiff(file_sc2)

    bins_sc1 = [2.600, 6.360, 8.662, 11.732, 15.799, 22.170]
    bins_sc2 = [3.913, 8.230, 10.721, 13.959, 18.276, 25.086]

    def reclassify(data, bins):
        classified = np.full(data.shape, np.nan)
        for i in range(len(bins) - 1):
            low, high = bins[i], bins[i+1]
            if i == len(bins) - 2:
                mask = (data >= low) & (data <= high + 0.1)
            else:
                mask = (data >= low) & (data < high)
            classified[mask] = i + 1
        return classified

    class_sc1 = reclassify(data_sc1, bins_sc1)
    class_sc2 = reclassify(data_sc2, bins_sc2)

    def calculate_area_percent(classified_data):
        valid = classified_data[~np.isnan(classified_data)]
        counts = np.bincount(valid.astype(int), minlength=6)[1:6]
        return (counts / counts.sum()) * 100

    pct_sc1 = calculate_area_percent(class_sc1)
    pct_sc2 = calculate_area_percent(class_sc2)

    transformer = pyproj.Transformer.from_crs(crs_sc1, "EPSG:4326", always_xy=True)
    center_x = (ext_sc1[0] + ext_sc1[1]) / 2
    center_y = (ext_sc1[2] + ext_sc1[3]) / 2

    def format_lon_fr(x, pos):
        lon, _ = transformer.transform(x, center_y)
        return f"{lon:.2f}°E"

    def format_lat_fr(y, pos):
        _, lat = transformer.transform(center_x, y)
        return f"{lat:.2f}°N"

    class_labels = ["Very Low", "Low", "Moderate", "High", "Very High"]
    hex_colors = ["#2b83ba", "#abdda4", "#ffffbf", "#fdae61", "#d7191c"]
    cmap_fr = mcolors.ListedColormap(hex_colors)
    norm_fr = mcolors.BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], ncolors=5)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), dpi=300)
    fig.subplots_adjust(bottom=0.15, wspace=0.20)

    im1 = axes[0].imshow(class_sc1, extent=ext_sc1, cmap=cmap_fr, norm=norm_fr, origin='upper')
    axes[0].set_title("(a) Scenario 1 (9 Factors)", fontsize=15, fontweight="bold", loc="left", pad=10)

    im2 = axes[1].imshow(class_sc2, extent=ext_sc2, cmap=cmap_fr, norm=norm_fr, origin='upper')
    axes[1].set_title("(b) Scenario 2 (11 Factors)", fontsize=15, fontweight="bold", loc="left", pad=10)

    for ax, pct in zip(axes, [pct_sc1, pct_sc2]):
        ax.set_xlabel("Longitude", fontsize=12, fontweight="bold", labelpad=5)
        ax.set_ylabel("Latitude", fontsize=12, fontweight="bold", labelpad=5)

        ax.xaxis.set_major_formatter(FuncFormatter(format_lon_fr))
        ax.yaxis.set_major_formatter(FuncFormatter(format_lat_fr))
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

        ax.tick_params(axis="x", labelsize=10)
        ax.tick_params(axis="y", labelsize=10)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.3)
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1.0)

        for idx, val in enumerate(pct):
            x_pos = 0.02 + (idx * 0.20)
            y_pos = -0.20
            ax.plot(x_pos, y_pos, marker='o', markersize=10, color=hex_colors[idx],
                    markeredgecolor='black', markeredgewidth=0.5, transform=ax.transAxes, clip_on=False)
            ax.text(x_pos + 0.04, y_pos, f"{val:.2f}%", transform=ax.transAxes,
                    ha='left', va='center', fontsize=11, fontweight='bold', clip_on=False)

    cbar_ax = fig.add_axes([0.35, 0.02, 0.3, 0.025])
    cbar = fig.colorbar(im1, cax=cbar_ax, orientation="horizontal", ticks=[1, 2, 3, 4, 5])
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(size=0)
    cbar.set_ticklabels(class_labels)
    cbar.set_label("Landslide Susceptibility Classes", fontsize=14, fontweight="bold", labelpad=10)

    # 🎯 บันทึกด้วยชื่อ _github
    output_map_fr = "classified_susceptibility_maps_comparison_FR_github.png"
    plt.savefig(output_map_fr, dpi=300, bbox_inches="tight")
    print(f"✅ Saved FR Map: {output_map_fr}")

    try:
        fig.canvas.draw()
        display(fig)
    except Exception:
        plt.show()

    plt.close(fig)
else:
    print("⚠️ Raster files (exclude1.tif, include1.tif) not found. Skipping FR Map rendering.")
